# Modelo de proyección y control · TATSA [TA01432] · Repuestos / Repuestos · cierre 2026

Genera **`Proyeccion_TA01432.xlsx`** con tres hojas:
- **RESUMEN:** mes a mes (octubre, noviembre, diciembre): **prometido de consumo, consumo real, compras (gasto de presupuesto), saldo disponible e inventario**, con gráficos; qué hay que hacer; cumplimiento por categoría; y **controles** que validan las cifras. Parámetros en celdas amarillas.
- **PLAN:** **todos los repuestos** del presupuesto, una fila por llave, con fórmulas. La columna **Accion** (primera) se filtra: COMPRAR, APURAR TRÁNSITO, ACELERAR CONSUMO, REVISAR PROYECCIÓN, ASIGNAR PRECIO. Las columnas van por bloques con un encabezado en la fila 1: repuesto (llave, código de la última compra y todos los códigos de la llave) → clasificación → consumo 12M (promedio y desviación) → precio → inventario → tránsito → análisis (proyección vs real, ajuste, cobertura y compra, inventario proyectado). El bloque DETALLE va plegado.
- **PROMETIDO:** las unidades del archivo del 13/10/2026 tal como se prometieron, con la llave actual.

**Cómo se trabaja**
1. Ejecutar `Tablas_Actual_BI` (kardex) → `Saldo_Disponible_Real` (saldo y tránsito) → **este notebook**.
2. En el Excel, editar **solo las celdas amarillas**: en PLAN `AJ_NOV`, `AJ_DIC` (unidades del mes), `PRECIO_MANUAL`, `COMENTARIO`; en RESUMEN los parámetros. Vacío = usar el cálculo. Todo (tablas y gráficos) se recalcula al instante.
3. Al volver a ejecutar el notebook **se conservan las ediciones** (se leen del Excel anterior antes de regenerarlo).

**Reglas**
- **El modelo detecta el mes en curso** (fecha de ejecución): un **mes cerrado** cuenta solo lo real (consumo del kardex y compras ejecutadas); el **mes en curso** = real a hoy + lo que falta del plan, y ahí se cuenta todo el tránsito (RQ con OC y sin OC); los **meses siguientes** = plan. En diciembre, noviembre pasa solo a cerrado.
- **Inventario al cierre de cada mes:** almacén TVL, almacén TATSA y tránsito por recibir por separado (real del kardex en meses cerrados, proyectado después).
- **Lo prometido:** solo las **unidades** por llave, tipo y mes del archivo del 13/10. Las llaves que cambiaron después se pasan a la llave actual por sus SKU (si se juntaron, se suma). `REP. DAR BAJA` no cuenta como consumo.
- **Un mismo repuesto en varias categorías:** el consumo real se separa por el **tipo de la OT** del kardex. Lo que salió en **OT preventivas** se compara con lo prometido como PREVENTIVO; lo que salió en **OT correctivas**, con lo prometido en los tipos correctivos de esa llave (correctivo programado, experto o estadístico). Lo que se consume en un lado sin promesa cae en NO PROYECTADO.

| Categoría | Tipos del archivo | Quién proyecta | Se compara con |
|---|---|---|---|
| PREVENTIVO | PREVENTIVO | Programador | Consumo en OT preventivas |
| CORRECTIVO PROGRAMADO | SERV_CORRECTIVO, CAMPAÑA, LLANTERIA | Programador / área de llantería | Consumo en OT correctivas |
| EXPERTO | CORRECTIVO IMPREDECIBLE | Criterio experto | Consumo en OT correctivas |
| ESTADÍSTICO | CORRECTIVO X PEDIDO | Promedio | Consumo en OT correctivas |
| NO PROYECTADO | SIN PROYECTAR, REP. DAR BAJA o sin promesa en ese lado | Se decide en AJ_NOV / AJ_DIC | El consumo sin promesa |

- **ABC‑XYZ calculado con el kardex** (consumo a buses 91 − 92 de las dos empresas, 12 meses previos al corte): ABC por valor (A ≤ 80 %, B ≤ 95 %), XYZ por CV mensual (X < 0,5, Y ≤ 1,0, Z > 1,0). Solo se usa el calculado (no el del archivo) y con él se sugiere el método del lado correctivo (A/B‑Z → EXPERTO; resto → ESTADÍSTICO).
- **PLAN tiene todos los repuestos** de las 14 familias con presupuesto, tránsito, consumo o stock (salvo material de Proyecto Chasis, que es de otros proyectos).
- **Cobertura:** stock **TVL** (primero, para dejar limpio ese almacén) → stock **TATSA** → **RQ con OC** → **RQ sin OC** → **compra nueva** (lo único que gasta saldo). Noviembre se cubre antes que diciembre.
- **Precio e IGV como Odoo, según la última compra real:** si la línea de la última OC tuvo IGV, el inventario se valoriza costo kardex + IGV; si no lo tuvo (importación o proveedor extranjero), sin IGV. Se informa la OC, el proveedor, su país y la moneda. Orden: precio manual → inventario (lo que sale del almacén) → última compra → compra en tránsito.

##### Librerías

In [ ]:
import os
import glob
import xmlrpc.client
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.formatting.rule import CellIsRule, FormulaRule
from openpyxl.chart import BarChart, LineChart, Reference
from openpyxl.chart.series import SeriesLabel

##### Parámetros

In [ ]:
ANIO = 2026
ARCHIVO_PROMETIDO = glob.glob(r"C:/Users/ADM/Documents/Data_Pool/Presupuesto/Ampliación Presupuestal 2026/Proyección_2026_19_OCT_Ajuste_Para_800K.xlsx")[0]
CORTE = pd.Timestamp(f"{ANIO}-09-30")            # Punto de partida: cierre de setiembre
HOY = pd.Timestamp.today().normalize()
CENTRO, SITUACION = "TA01432", "REPUESTOS / REPUESTOS"
FAMILIAS_14 = ["MECANICO", "ELECTRONICA", "ELECTRICIDAD", "LUBRICANTES", "GOMERIA", "CARROCERIA", "FILTROS", "RODAMIENTOS",
               "FERRETERIA EN GENERAL", "INSUMOS DE PINTURAS", "FAJAS", "GAS", "MANGUERAS", "PERNOS Y HUACHAS"]
CATEGORIA = {"PREVENTIVO": "PREVENTIVO", "SERV_CORRECTIVO": "CORRECTIVO PROGRAMADO", "CAMPAÑA": "CORRECTIVO PROGRAMADO", "LLANTERIA": "CORRECTIVO PROGRAMADO",
             "CORRECTIVO IMPREDECIBLE": "EXPERTO", "CORRECTIVO X PEDIDO": "ESTADÍSTICO"}
QUIEN = {"PREVENTIVO": "Programador", "SERV_CORRECTIVO": "Programador", "CAMPAÑA": "Programador", "LLANTERIA": "Área de llantería",
         "CORRECTIVO IMPREDECIBLE": "Criterio experto", "CORRECTIVO X PEDIDO": "Promedio"}
CATEGORIAS = ["PREVENTIVO", "CORRECTIVO PROGRAMADO", "EXPERTO", "ESTADÍSTICO", "NO PROYECTADO"]
ABC_CORTES, XYZ_CORTES = (0.80, 0.95), (0.50, 1.00)
PARAMETROS_INICIALES = {"IGV": 0.18, "MESES_SOBRESTOCK": 6, "UMBRAL_DESVIO": 0.30, "MONTO_MIN_REVISAR": 1000}
HISTORIA_OC = "2023-01-01"
USUARIO = "JIM"
ARCHIVO_DIM_PRODUCTO = "Información_Proyecto_BI/Dim_Producto.parquet"
ARCHIVO_KARDEX = "Información_Proyecto_BI/Fact_Kardex.parquet"
ARCHIVO_SALDO = "Saldo_Disponible_Real_2024_2026.xlsx"
ARCHIVO_EXCEL = "Proyeccion_TA01432.xlsx"
# Meses del cierre: los anteriores al mes en curso están cerrados (solo cuenta lo real); el mes en curso = real a hoy + lo que falta; los siguientes = plan
MES_ACTUAL = HOY.month if HOY.year == ANIO else 13
assert MES_ACTUAL in (11, 12), "El modelo está pensado para correrse en noviembre o diciembre del año del cierre"
MESES = {10: "OCT", 11: "NOV", 12: "DIC"}
NOMBRE_MES = {10: "Octubre", 11: "Noviembre", 12: "Diciembre"}
FIN_MES = {9: "30SET", 10: "31OCT", 11: "30NOV", 12: "31DIC"}
CERRADOS = [m for m in MESES if m < MES_ACTUAL]
ESTADO = {m: "real" if m < MES_ACTUAL else ("real a hoy + falta" if m == MES_ACTUAL else "plan") for m in MESES}
TXT_CERRADOS = "–".join(MESES[m] for m in CERRADOS) if len(CERRADOS) < 3 else "OCT–DIC"
print(f"Corte {CORTE.date()} | hoy {HOY.date()} | mes en curso: {NOMBRE_MES[MES_ACTUAL]} | cerrados: {TXT_CERRADOS} | prometido: {os.path.basename(ARCHIVO_PROMETIDO)}")

##### Conexión (solo lectura)

In [ ]:
load_dotenv()
url, db = os.getenv("URL_ODOO"), os.getenv("BD_ODOO")
username, password = os.getenv(f"USER_{USUARIO}"), os.getenv(f"PASSWORD_{USUARIO}")
uid = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common").authenticate(db, username, password, {})
models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object", allow_none=True)
print(f"uid conectado : {uid}")


def odoo(modelo, metodo, argumentos, opciones=None, empresas=None):
    opciones = dict(opciones or {})
    contexto = {"lang": "es_PE", "tz": "America/Lima", **opciones.pop("context", {})}
    if empresas:
        contexto["allowed_company_ids"] = empresas
    opciones["context"] = contexto
    return models.execute_kw(db, uid, password, modelo, metodo, argumentos, opciones)


def leer_todo(modelo, dominio, campos, empresas, lote=5000, contexto=None):
    filas, desplazamiento = [], 0
    while True:
        bloque = odoo(modelo, "search_read", [dominio], {"fields": campos, "limit": lote, "offset": desplazamiento, "order": "id", "context": contexto or {}}, empresas)
        filas += bloque
        if len(bloque) < lote:
            return filas
        desplazamiento += lote


def m2o_id(valor):
    return valor[0] if isinstance(valor, list) and valor else None


def m2o_nombre(valor):
    return valor[1] if isinstance(valor, list) and len(valor) == 2 else None

##### 1. Lo prometido: unidades del 13/10 llevadas a la llave actual y separadas por lado (preventivo / correctivo)

In [ ]:
dim = pd.read_parquet(ARCHIVO_DIM_PRODUCTO)
dim["Llave"] = dim["Cod_Grupo_Sku"].fillna(dim["Cod_Producto"])
LLAVE_DE = dim.drop_duplicates("Cod_Producto").set_index("Cod_Producto")["Llave"].to_dict()
FAMILIA = dim.groupby("Llave")["Familia"].agg(lambda s: s.value_counts().index[0]).to_dict()
NOMBRE = dim.drop_duplicates("Llave").set_index("Llave")["Producto_Grupo"].to_dict()

proy = pd.read_excel(ARCHIVO_PROMETIDO, sheet_name="Proyección", header=1).iloc[:, 1:7]
proy.columns = ["Llave_13OCT", "Producto", "Tipo", "OCT", "NOV", "DIC"]
proy = proy.dropna(subset=["Llave_13OCT"])
proy[["OCT", "NOV", "DIC"]] = proy[["OCT", "NOV", "DIC"]].fillna(0.0)
detalle = pd.read_excel(ARCHIVO_PROMETIDO, sheet_name="Detalle", header=2).set_index("LLAVE")
UND_ARCHIVO = float(detalle[["OCT", "NOV", "DIC"]].values.sum())
print(f"Unidades prometidas: {proy[['OCT', 'NOV', 'DIC']].sum().round(2).to_dict()} | cuadra con la hoja Detalle: "
      f"{abs(proy[['OCT', 'NOV', 'DIC']].values.sum() - UND_ARCHIVO) < 0.01}")

# Llave del 13/10 → llave actual, por los SKU del maestro que trae el propio archivo
maestro_13 = pd.read_excel(ARCHIVO_PROMETIDO, sheet_name="Producto")[["Cod_Producto", "Cod_Grupo_Sku"]].dropna(subset=["Cod_Producto"])
maestro_13["Llave_13OCT"] = maestro_13["Cod_Grupo_Sku"].fillna(maestro_13["Cod_Producto"])
maestro_13["Llave_Actual"] = maestro_13["Cod_Producto"].map(LLAVE_DE).fillna(maestro_13["Cod_Producto"])
mapa, nota = {}, {}
for ll, g in maestro_13[maestro_13["Llave_13OCT"].isin(proy["Llave_13OCT"])].groupby("Llave_13OCT"):
    conteo = g["Llave_Actual"].value_counts()
    mapa[ll] = conteo.index[0]
    if len(conteo) > 1:
        nota[ll] = f"Llave dividida en {', '.join(conteo.index)}: revisar"
    elif conteo.index[0] != ll:
        nota[ll] = f"Hoy es {conteo.index[0]}"
proy["Llave"] = proy["Llave_13OCT"].map(mapa).fillna(proy["Llave_13OCT"])
juntas = proy.groupby("Llave")["Llave_13OCT"].nunique()
for ll13, ll in mapa.items():
    if juntas.get(ll, 1) > 1 and "dividida" not in nota.get(ll13, ""):
        nota[ll13] = f"Agrupada en {ll} (se suma con {', '.join(sorted(set(proy.loc[proy['Llave'] == ll, 'Llave_13OCT']) - {ll13}))})"
proy["Nota_Llave"] = proy["Llave_13OCT"].map(nota).fillna("")
proy["Cuenta_Como_Consumo"] = np.where(proy["Tipo"] == "REP. DAR BAJA", "No: repuesto para dar de baja", "Sí")
proy["Categoria"] = proy["Tipo"].map(CATEGORIA).fillna("NO PROYECTADO")
print("Cambios de llave desde el 13/10:", nota)

consumo = proy.copy()
consumo.loc[consumo["Tipo"] == "REP. DAR BAJA", ["OCT", "NOV", "DIC"]] = 0.0
consumo["Lado"] = np.where(consumo["Tipo"] == "PREVENTIVO", "PREV", "CORR")
prom = consumo.groupby("Llave").agg(Tipo=("Tipo", lambda s: " + ".join(sorted(set(s)))), Llave_13OCT=("Llave_13OCT", lambda s: ", ".join(sorted(set(s)))),
                                    Nota_Llave=("Nota_Llave", lambda s: " · ".join(sorted({x for x in s if x}))))
for m in ["OCT", "NOV", "DIC"]:
    lado = consumo.pivot_table(index="Llave", columns="Lado", values=m, aggfunc="sum").reindex(columns=["PREV", "CORR"]).fillna(0.0)
    prom[f"PY_{m}_PREV"], prom[f"PY_{m}_CORR"] = lado["PREV"], lado["CORR"]
# Categoría de cada lado: preventivo si tiene tipo PREVENTIVO; correctivo = el tipo correctivo con más unidades
prom["Cat_Prev"] = np.where(prom.index.isin(consumo.loc[consumo["Tipo"] == "PREVENTIVO", "Llave"]), "PREVENTIVO", "NO PROYECTADO")
corr = consumo[(consumo["Lado"] == "CORR") & consumo["Tipo"].isin(CATEGORIA)]
tipo_corr = corr.assign(U=corr[["OCT", "NOV", "DIC"]].sum(axis=1)).groupby(["Llave", "Tipo"])["U"].sum().reset_index().sort_values("U", ascending=False).drop_duplicates("Llave").set_index("Llave")["Tipo"]
prom["Cat_Corr"] = tipo_corr.map(CATEGORIA).reindex(prom.index).fillna("NO PROYECTADO")
prom["Proyecta"] = [" / ".join(sorted({QUIEN[t] for t in tt.split(" + ") if t in QUIEN})) or "—" for tt in prom["Tipo"]]
print(f"Llaves del 13/10: {proy['Llave_13OCT'].nunique()} → llaves actuales: {len(prom)} | con promesa preventiva y correctiva a la vez: "
      f"{int(((prom[['PY_OCT_PREV', 'PY_NOV_PREV', 'PY_DIC_PREV']].sum(axis=1) > 0) & (prom[['PY_OCT_CORR', 'PY_NOV_CORR', 'PY_DIC_CORR']].sum(axis=1) > 0)).sum())}")

##### 2. Datos: consumo real (separado por tipo de OT), stock, tránsito y saldo

In [ ]:
kardex = pd.read_parquet(ARCHIVO_KARDEX, columns=["Fecha", "Fecha_Completa", "Empresa", "Almacen", "Cod_Producto", "Cod_Movimiento", "Tipo_Mantto",
                                                  "Cant_Ingreso", "Cant_Salida", "Valor_Ingreso", "Valor_Salida", "Cant_Saldo", "Valor_Saldo"])
kardex["Llave"] = kardex["Cod_Producto"].map(LLAVE_DE).fillna(kardex["Cod_Producto"])
bus = kardex[kardex["Cod_Movimiento"].isin([91, 92])].assign(Und=lambda d: d["Cant_Salida"] - d["Cant_Ingreso"], Valor=lambda d: d["Valor_Salida"] - d["Valor_Ingreso"])
bus["Periodo"] = bus["Fecha"].dt.to_period("M")
bus["Lado"] = np.where(bus["Tipo_Mantto"] == "Preventivo", "PREV", "CORR")       # sin tipo (0,3 %) = correctivo
REAL = bus[bus["Periodo"].isin([pd.Period(f"{ANIO}-{m}", "M") for m in MESES])].groupby(["Llave", "Periodo", "Lado"])["Und"].sum()
ventana = pd.period_range(pd.Period(CORTE, "M") - 11, pd.Period(CORTE, "M"), freq="M")
b12 = bus[bus["Periodo"].isin(ventana)]
SERIE_12M = b12.groupby(["Llave", "Periodo"])["Und"].sum().unstack(fill_value=0.0).reindex(columns=ventana, fill_value=0.0).clip(lower=0)
VALOR_12M = b12.groupby("Llave")["Valor"].sum().clip(lower=0)
LLAVES_BUS = set(bus["Llave"])
KARDEX_OCT = bus[bus["Periodo"] == pd.Period(f"{ANIO}-10", "M")].groupby("Llave")["Und"].sum()


def stock_al(fecha):
    """Stock y costo (kardex, sin IGV) por empresa y llave al final del día: último movimiento de cada almacén interno, sumados."""
    u = (kardex[kardex["Fecha"] <= fecha].sort_values("Fecha_Completa", kind="stable")
         .groupby(["Empresa", "Almacen", "Cod_Producto"]).last().reset_index())
    u = u[u["Cant_Saldo"] > 0]
    return u.groupby(["Empresa", "Llave"])[["Cant_Saldo", "Valor_Saldo"]].sum()


ST_HOY = stock_al(HOY)
ST_FIN = {m: stock_al(pd.Timestamp(f"{ANIO}-{m:02d}-01") + pd.offsets.MonthEnd(0)) for m in [9] + CERRADOS}   # stock real al cierre de cada mes cerrado

filtro = lambda d: d[(d["Empresa"] == "TATSA") & d["Cuenta Analítica"].str.contains(CENTRO, na=False) & (d["Situación presupuestaria"] == SITUACION)].copy()
saldo_sku = filtro(pd.read_excel(ARCHIVO_SALDO, sheet_name="SALDO", engine="calamine"))
saldo_sku = saldo_sku[saldo_sku["Periodo"] == ANIO]
oc_pend = filtro(pd.read_excel(ARCHIVO_SALDO, sheet_name="OC_PENDIENTE_ENTREGA", engine="calamine"))
rq_soc = filtro(pd.read_excel(ARCHIVO_SALDO, sheet_name="RQ_SIN_OC", engine="calamine"))
for d in (saldo_sku, oc_pend, rq_soc):
    d["Llave"] = d["Cod_Producto"].map(LLAVE_DE).fillna(d["Cod_Producto"])

res_saldo = filtro(pd.read_excel(ARCHIVO_SALDO, sheet_name="RESUMEN", engine="calamine"))
res_saldo = res_saldo[res_saldo["Periodo"] == ANIO]
# Un mes cerrado solo cuenta lo comprado (recibido o facturado); el tránsito (RQ con OC y sin OC, de cualquier mes) va al mes en curso
gasto = filtro(pd.read_excel(ARCHIVO_SALDO, sheet_name="GASTO_MES", engine="calamine"))
gasto = gasto[gasto["Periodo"] == ANIO]
EJECUTADO = gasto[gasto["Fuente"].isin(["Compras", "Gasto_Sin_OC"])].groupby("Mes")["Gasto"].sum().reindex(range(1, 13), fill_value=0.0)
TRANSITO_COC = float(gasto.loc[gasto["Fuente"] == "RQ_COC", "Gasto"].sum())
TRANSITO_SOC = float(gasto.loc[gasto["Fuente"] == "RQ_SOC", "Gasto"].sum())
# Compras ejecutadas por mes del cierre: lo fechado después del mes en curso (si hubiera) se suma al mes en curso
EJEC_MES = {m: float(EJECUTADO[m]) if m < MES_ACTUAL else (float(EJECUTADO[EJECUTADO.index >= m].sum()) if m == MES_ACTUAL else 0.0) for m in MESES}
SALDO = {"Presupuesto": float(res_saldo["Presupuesto"].sum()), "Ejec_ENE_SET": float(EJECUTADO[EJECUTADO.index <= 9].sum()),
         "Transito_COC": TRANSITO_COC, "Transito_SOC": TRANSITO_SOC, "Saldo_Hoy": float(res_saldo["Saldo"].sum()), "Saldo_Odoo": float(res_saldo["Disponible_Odoo"].sum())}
SALDO["Saldo_30SET"] = SALDO["Presupuesto"] - SALDO["Ejec_ENE_SET"]
# Referencia: al 30/09 descontando también los RQ de ene–set que aún están en tránsito
SALDO["Saldo_30SET_Con_Transito"] = SALDO["Presupuesto"] - float(gasto.loc[gasto["Mes"] <= 9, "Gasto"].sum())
CONTROL_SALDO = abs(SALDO["Presupuesto"] - EJECUTADO.sum() - TRANSITO_COC - TRANSITO_SOC - SALDO["Saldo_Hoy"])
FECHA_SALDO = pd.Timestamp.fromtimestamp(os.path.getmtime(ARCHIVO_SALDO)).strftime("%d/%m/%Y %H:%M")
print({k: round(v, 2) for k, v in SALDO.items()}, "| control: presupuesto − ejecutado − tránsito = saldo hoy:", CONTROL_SALDO < 0.01)

##### 3. Todos los repuestos del presupuesto (PLAN) y ABC‑XYZ calculado

In [ ]:
stock_tatsa = {ll for (e, ll) in ST_HOY.index if e == "Tatsa"}
stock_tvl = {ll for (e, ll) in ST_HOY.index if e == "Tvl"}
reales = set(REAL[REAL > 0].index.get_level_values(0))
origen = {"Prometido": set(prom.index), "Presupuesto TA01432": set(saldo_sku["Llave"]), "Tránsito": set(oc_pend["Llave"]) | set(rq_soc["Llave"]),
          "Consumo OCT/NOV": reales, "Stock TATSA": stock_tatsa, "Stock TVL (repuesto de buses)": stock_tvl & LLAVES_BUS}
TIPO_PROYECTO = dim.groupby("Llave")["Tipo_Proyecto"].agg(lambda s: s.value_counts().index[0]).to_dict()
chasis = {ll for ll in set().union(*origen.values()) if TIPO_PROYECTO.get(ll) == "Proyecto Chasis"} - origen["Prometido"]
universo = sorted(ll for ll in set().union(*origen.values()) - chasis if FAMILIA.get(ll) in FAMILIAS_14)
print(f"PLAN: {len(universo)} llaves | prometidas {len(set(universo) & origen['Prometido'])} | no proyectadas {len(set(universo) - origen['Prometido'])}"
      f" | excluidas por Proyecto Chasis: {len(chasis)}")

plan = pd.DataFrame({"Llave": universo}).join(prom, on="Llave")
plan["Producto"] = plan["Llave"].map(NOMBRE).fillna(plan["Llave"].map(proy.drop_duplicates("Llave").set_index("Llave")["Producto"]))
plan["Familia"] = plan["Llave"].map(FAMILIA)
plan["Tipo"] = plan["Tipo"].fillna("No estaba en la proyección")
for c in ["Cat_Prev", "Cat_Corr"]:
    plan[c] = plan[c].fillna("NO PROYECTADO")
plan["Proyecta"] = plan["Proyecta"].fillna("—")
plan["Nota_Llave"] = plan["Nota_Llave"].fillna("")
for m in ["OCT", "NOV", "DIC"]:
    for lado in ["PREV", "CORR"]:
        plan[f"PY_{m}_{lado}"] = plan[f"PY_{m}_{lado}"].fillna(0.0)
    plan[f"PY_{m}"] = plan[f"PY_{m}_PREV"] + plan[f"PY_{m}_CORR"]
for nm, m in MESES.items():
    per = pd.Period(f"{ANIO}-{nm}", "M")
    for lado in ["PREV", "CORR"]:
        plan[f"REAL_{m}_{lado}"] = [REAL.get((ll, per, lado), 0.0) for ll in plan["Llave"]]
    plan[f"REAL_{m}"] = plan[f"REAL_{m}_PREV"] + plan[f"REAL_{m}_CORR"]
plan["Metodo"] = np.where(plan["Cat_Corr"] != "NO PROYECTADO", plan["Cat_Corr"], plan["Cat_Prev"])
plan["Origen"] = [", ".join(k for k, v in origen.items() if ll in v) for ll in plan["Llave"]]
plan["Prom_Mes_12M"] = plan["Llave"].map(SERIE_12M.sum(axis=1) / 12).fillna(0.0)
for alm, col in [("Tvl", "TVL"), ("Tatsa", "TATSA")]:
    plan[f"Stock_{col}"] = [ST_HOY["Cant_Saldo"].get((alm, ll), 0.0) for ll in plan["Llave"]]
    plan[f"Costo_{col}"] = [ST_HOY["Valor_Saldo"][(alm, ll)] / ST_HOY["Cant_Saldo"][(alm, ll)] if (alm, ll) in ST_HOY.index else np.nan for ll in plan["Llave"]]
STOCK_FIN = [f"Stock_{col}_{FIN_MES[m]}" for m in ST_FIN for col in ("TVL", "TATSA")]
for m, st in ST_FIN.items():
    for alm, col in [("Tvl", "TVL"), ("Tatsa", "TATSA")]:
        plan[f"Stock_{col}_{FIN_MES[m]}"] = [st["Cant_Saldo"].get((alm, ll), 0.0) for ll in plan["Llave"]]

# ABC-XYZ con el kardex (12 meses previos al corte), dentro del universo de PLAN
valor = pd.Series(plan["Llave"].map(VALOR_12M).fillna(0.0).values, index=plan["Llave"]).sort_values(ascending=False)
acum = valor.cumsum() / valor.sum()
abc = pd.Series(np.where(valor <= 0, "-", np.where(acum <= ABC_CORTES[0], "A", np.where(acum <= ABC_CORTES[1], "B", "C"))), index=valor.index)
serie = SERIE_12M.reindex(plan["Llave"]).fillna(0.0)
cv = (serie.std(axis=1, ddof=0) / serie.mean(axis=1)).replace([np.inf, -np.inf], np.nan)
xyz = pd.Series(np.where(cv.isna(), "-", np.where(cv < XYZ_CORTES[0], "X", np.where(cv <= XYZ_CORTES[1], "Y", "Z"))), index=cv.index)
plan["ABC_XYZ"] = [("SIN CONSUMO 12M" if abc[ll] == "-" else abc[ll] + xyz[ll]) for ll in plan["Llave"]]
plan["Valor_Consumo_12M"] = plan["Llave"].map(valor)
plan["CV_12M"] = plan["Llave"].map(cv)
plan["Desv_Mes_12M"] = plan["Llave"].map(serie.std(axis=1, ddof=0))
CODIGOS = dim.dropna(subset=["Cod_Producto"]).groupby("Llave")["Cod_Producto"].agg(lambda s: ", ".join(sorted(set(s.astype(str)))))
plan["Codigos_Llave"] = plan["Llave"].map(CODIGOS).fillna(plan["Llave"])
sug = np.where(plan["ABC_XYZ"] == "SIN CONSUMO 12M", "SIN CONSUMO 12M",
               np.where(plan["ABC_XYZ"].str[0].isin(["A", "B"]) & plan["ABC_XYZ"].str.endswith("Z"), "EXPERTO", "ESTADÍSTICO"))
plan["Metodo_Sugerido"] = np.where(plan["Cat_Corr"] == "CORRECTIVO PROGRAMADO", "CORRECTIVO PROGRAMADO", sug)
plan["Revisar_Metodo"] = np.where(plan["Cat_Corr"].isin(["EXPERTO", "ESTADÍSTICO"]) & (plan["Cat_Corr"] != plan["Metodo_Sugerido"]), "Sí", "No")
print("ABC-XYZ (kardex):", plan["ABC_XYZ"].value_counts().to_dict(), "| método experto/estadístico a revisar:", int((plan["Revisar_Metodo"] == "Sí").sum()))

g = oc_pend.groupby("Llave")
plan["RQ_COC_Und"] = plan["Llave"].map(g["Cant_Pendiente"].sum()).fillna(0.0)
plan["OC_Pendiente"] = plan["Llave"].map(g["Cod_OC"].agg(lambda s: ", ".join(sorted(set(s)))))
plan["Comprador"] = plan["Llave"].map(g["Comprador"].agg(lambda s: ", ".join(sorted(set(s.dropna())))))
plan["Fecha_Prevista"] = plan["Llave"].map(g["Fecha_Entrega_Prevista"].max())
plan["Dias_Atraso"] = plan["Llave"].map(g["Dias_Atraso"].max()).fillna(0.0)
g = rq_soc.groupby("Llave")
plan["RQ_SOC_Und"] = plan["Llave"].map(g["Cant_Pendiente"].sum()).fillna(0.0)
plan["RQ_Sin_OC"] = plan["Llave"].map(g["Cod_RQ"].agg(lambda s: ", ".join(sorted(set(s)))))
ptr = oc_pend.groupby("Llave")[["Valor_Pendiente", "Cant_Pendiente"]].sum()
PRECIO_TRANSITO = (ptr["Valor_Pendiente"] / ptr["Cant_Pendiente"]).replace([np.inf], np.nan)

##### 4. Última compra real: precio, IGV e importación

Se toma la última línea de OC confirmada de cada llave (las dos empresas):
- **Con IGV:** la línea tiene IGV (total ÷ subtotal ≈ 1,18). Es lo que decide si al costo del inventario se le suma IGV.
- **Importación:** la OC está marcada como importación **o** el proveedor es extranjero (país distinto de Perú).

In [ ]:
productos = leer_todo("product.product", [], ["default_code", "uom_id"], [1, 2], contexto={"active_test": False})
PRODUCTO = {p["id"]: p for p in productos}
UDM = {u["id"]: (u["factor"] or 1.0, m2o_id(u["category_id"])) for u in odoo("uom.uom", "search_read", [[]], {"fields": ["factor", "category_id"], "context": {"active_test": False}})}


def a_udm(cantidad, udm, product_id):
    destino = m2o_id(PRODUCTO.get(product_id, {}).get("uom_id"))
    if udm in UDM and destino in UDM and udm != destino and UDM[udm][1] == UDM[destino][1]:
        return cantidad / UDM[udm][0] * UDM[destino][0]
    return cantidad


filas, socios = [], set()
for empresa in [2, 1]:
    ordenes = {o["id"]: o for o in leer_todo("purchase.order", [["company_id", "=", empresa], ["state", "in", ["purchase", "done"]], ["date_order", ">=", HISTORIA_OC]],
                                             ["name", "currency_rate", "currency_id", "date_approve", "date_order", "importacion_check", "partner_id"], [empresa])}
    socios |= {m2o_id(o["partner_id"]) for o in ordenes.values() if o["partner_id"]}
    for l in leer_todo("purchase.order.line", [["order_id", "in", list(ordenes)], ["product_id", "!=", False]],
                       ["order_id", "product_id", "product_uom", "product_qty", "price_total", "price_subtotal"], [empresa]):
        o, pid = ordenes[m2o_id(l["order_id"])], m2o_id(l["product_id"])
        cant = a_udm(l["product_qty"] or 0.0, m2o_id(l["product_uom"]), pid)
        if cant > 0 and l["price_total"]:
            cod = PRODUCTO.get(pid, {}).get("default_code")
            filas.append({"OC": o["name"], "Fecha": (o["date_approve"] or o["date_order"])[:10], "Llave": LLAVE_DE.get(cod, cod), "Cod": cod, "Marca_Import": bool(o["importacion_check"]),
                          "Partner": m2o_id(o["partner_id"]), "Proveedor": m2o_nombre(o["partner_id"]), "Moneda": m2o_nombre(o["currency_id"]),
                          "Ratio_IGV": (l["price_total"] / l["price_subtotal"]) if l["price_subtotal"] else 1.0,
                          "Precio": l["price_total"] / (o["currency_rate"] or 1.0) / cant})
PAIS = {}
for i in range(0, len(socios), 1000):
    lote = list(socios)[i:i + 1000]
    PAIS.update({p["id"]: m2o_nombre(p["country_id"]) or "" for p in odoo("res.partner", "read", [lote], {"fields": ["country_id"], "context": {"active_test": False}}, [1, 2])})
compras = pd.DataFrame(filas).sort_values("Fecha")
compras["Pais"] = compras["Partner"].map(PAIS).fillna("")
compras["Extranjero"] = ~compras["Pais"].isin(["Perú", "Peru", ""])
ultima = compras.groupby("Llave").last()
plan["Ultima_OC"] = plan["Llave"].map(ultima["OC"])
plan["Cod_Ult_Compra"] = plan["Llave"].map(ultima["Cod"])
plan["Fecha_Ult_Compra"] = plan["Llave"].map(ultima["Fecha"])
plan["Proveedor_Ult"] = plan["Llave"].map(ultima["Proveedor"])
plan["Pais_Proveedor"] = plan["Llave"].map(ultima["Pais"])
plan["Moneda_Ult"] = plan["Llave"].map(ultima["Moneda"])
plan["Importacion"] = np.where(plan["Llave"].map(ultima["Marca_Import"] | ultima["Extranjero"]).fillna(False).astype(bool), "Sí", "No")
plan["Compra_Con_IGV"] = np.where(plan["Ultima_OC"].isna(), "Sin compras", np.where(plan["Llave"].map(ultima["Ratio_IGV"]).fillna(1.18) > 1.10, "Sí", "No"))
plan["Precio_Compra"] = plan["Llave"].map(ultima["Precio"]).fillna(plan["Llave"].map(PRECIO_TRANSITO))
plan["Costo_Inventario"] = plan["Costo_TVL"].where(plan["Stock_TVL"] > 0).fillna(plan["Costo_TATSA"].where(plan["Stock_TATSA"] > 0))
print(pd.crosstab(plan["Importacion"], plan["Compra_Con_IGV"]).to_string())
print(f"Con precio de compra: {plan['Precio_Compra'].notna().sum()} | con costo de inventario: {plan['Costo_Inventario'].notna().sum()} | "
      f"sin ningún precio: {(plan['Precio_Compra'].isna() & plan['Costo_Inventario'].isna()).sum()}")

##### 5. Conservar lo editado en el Excel anterior

In [ ]:
EDITABLES = ["AJ_NOV", "AJ_DIC", "PRECIO_MANUAL", "COMENTARIO"]
PARAM_CELDA = {"IGV": "K5", "MESES_SOBRESTOCK": "K6", "UMBRAL_DESVIO": "K7", "MONTO_MIN_REVISAR": "K8"}
PARAM_CELDA_ANTES = {"IGV": "I5", "MESES_SOBRESTOCK": "I6", "UMBRAL_DESVIO": "I7", "MONTO_MIN_REVISAR": "I8"}
previo, parametros = {}, dict(PARAMETROS_INICIALES)
if os.path.exists(ARCHIVO_EXCEL):
    try:
        wb_prev = load_workbook(ARCHIVO_EXCEL)
        if "PLAN" in wb_prev.sheetnames:
            h = wb_prev["PLAN"]
            fila_cab = 2 if "Llave" in [c.value for c in h[2]] else 1          # antes el encabezado estaba en la fila 1
            cab = [c.value for c in h[fila_cab]]
            if "Llave" in cab and all(e in cab for e in EDITABLES):
                il = cab.index("Llave")
                for fila in h.iter_rows(min_row=fila_cab + 1, values_only=True):
                    if fila[il] and any(fila[cab.index(e)] not in (None, "") for e in EDITABLES):
                        previo[LLAVE_DE.get(fila[il], fila[il])] = {e: fila[cab.index(e)] for e in EDITABLES}
        if "RESUMEN" in wb_prev.sheetnames:
            hr_prev = wb_prev["RESUMEN"]
            celdas = PARAM_CELDA if hr_prev["J4"].value == "Parámetros" else PARAM_CELDA_ANTES
            for k, celda_ in celdas.items():
                if isinstance(hr_prev[celda_].value, (int, float)):
                    parametros[k] = hr_prev[celda_].value
    except PermissionError:
        raise SystemExit(f"Cierra {ARCHIVO_EXCEL} en Excel y vuelve a ejecutar.")
for e in EDITABLES:
    plan[e] = plan["Llave"].map(lambda ll: previo.get(ll, {}).get(e))
perdidas = sorted(set(previo) - set(plan["Llave"]))
print(f"Ediciones conservadas: {len(previo) - len(perdidas)} llaves" + (f" | llaves editadas que ya no están: {perdidas}" if perdidas else "") + f" | parámetros: {parametros}")

##### 6. Cálculo en Python (control: el Excel debe dar lo mismo)

In [ ]:
P = parametros
c = plan.copy()
num = lambda s: pd.to_numeric(s, errors="coerce")
c["PLAN_NOV"] = num(c["AJ_NOV"]).fillna(c["PY_NOV"])
c["PLAN_DIC"] = num(c["AJ_DIC"]).fillna(c["PY_DIC"])
for m in (11, 12):                                   # mes cerrado: nada pendiente; en curso: lo que falta; futuro: el plan
    M = MESES[m]
    c[f"Pend_{M}"] = 0.0 if m < MES_ACTUAL else ((c[f"PLAN_{M}"] - c[f"REAL_{M}"]).clip(lower=0) if m == MES_ACTUAL else c[f"PLAN_{M}"])
c["Pend"] = c["Pend_NOV"] + c["Pend_DIC"]
c["Usa_TVL"] = np.minimum(c["Stock_TVL"], c["Pend"])
c["Usa_TATSA"] = np.minimum(c["Stock_TATSA"], c["Pend"] - c["Usa_TVL"])
c["Usa_COC"] = np.minimum(c["RQ_COC_Und"], c["Pend"] - c["Usa_TVL"] - c["Usa_TATSA"])
c["Usa_SOC"] = np.minimum(c["RQ_SOC_Und"], c["Pend"] - c["Usa_TVL"] - c["Usa_TATSA"] - c["Usa_COC"])
c["Cobertura"] = c["Stock_TVL"] + c["Stock_TATSA"] + c["RQ_COC_Und"] + c["RQ_SOC_Und"]
c["Compra_NOV_Und"] = (c["Pend_NOV"] - c["Cobertura"]).clip(lower=0)
c["Compra_Und"] = c["Pend"] - c["Usa_TVL"] - c["Usa_TATSA"] - c["Usa_COC"] - c["Usa_SOC"]
c["Compra_DIC_Und"] = c["Compra_Und"] - c["Compra_NOV_Und"]
factor = np.where(c["Compra_Con_IGV"] == "No", 1.0, 1 + P["IGV"])
c["P_Inv"] = num(c["PRECIO_MANUAL"]).fillna(c["Costo_Inventario"] * factor).fillna(c["Precio_Compra"]).fillna(0.0)
c["P_Com"] = num(c["PRECIO_MANUAL"]).fillna(c["Precio_Compra"]).fillna(c["Costo_Inventario"] * factor).fillna(0.0)
c["Inv_TVL_DIC"] = c["Stock_TVL"] - c["Usa_TVL"]
c["Inv_TATSA_DIC"] = c["Stock_TATSA"] - c["Usa_TATSA"]
c["Trans_DIC"] = c["RQ_COC_Und"] + c["RQ_SOC_Und"] - c["Usa_COC"] - c["Usa_SOC"]
control_py = {"Compra_NOV": (c["Compra_NOV_Und"] * c["P_Com"]).sum(), "Compra_DIC": (c["Compra_DIC_Und"] * c["P_Com"]).sum(),
              "Consumo_real_OCT": (c["REAL_OCT"] * c["P_Inv"]).sum(), "Prometido_OCT": (c["PY_OCT"] * c["P_Inv"]).sum(),
              "Inventario_hoy": ((c["Stock_TVL"] + c["Stock_TATSA"]) * c["P_Inv"]).sum(), "Usa_almacen": ((c["Usa_TVL"] + c["Usa_TATSA"]) * c["P_Inv"]).sum(),
              "Inv_TVL_31DIC": (c["Inv_TVL_DIC"] * c["P_Inv"]).sum(), "Inv_TATSA_31DIC": (c["Inv_TATSA_DIC"] * c["P_Inv"]).sum(),
              "Transito_sin_consumir_31DIC": (c["Trans_DIC"] * c["P_Com"]).sum()}
control_py["Saldo_31DIC"] = SALDO["Saldo_Hoy"] - control_py["Compra_NOV"] - control_py["Compra_DIC"]
control_py["Inv_total_31DIC"] = control_py["Inv_TVL_31DIC"] + control_py["Inv_TATSA_31DIC"] + control_py["Transito_sin_consumir_31DIC"]
# Controles con los datos de origen (van al RESUMEN)
CONTROLES_PY = [
    ("Unidades prometidas cargadas = archivo de octubre", UND_ARCHIVO, float(proy[["OCT", "NOV", "DIC"]].values.sum())),
    ("Saldo hoy = presupuesto − compras ejecutadas − tránsito (Excel del saldo)", SALDO["Saldo_Hoy"],
     SALDO["Presupuesto"] - float(EJECUTADO.sum()) - SALDO["Transito_COC"] - SALDO["Transito_SOC"]),
    ("Consumo real de octubre en PLAN = kardex de esas llaves", float(KARDEX_OCT.reindex(plan["Llave"]).fillna(0).sum()), float(plan["REAL_OCT"].sum())),
    ("Stock de hoy en PLAN = kardex de hoy de esas llaves (TVL + TATSA)",
     float(sum(ST_HOY["Cant_Saldo"].get((a, ll), 0.0) for a in ("Tvl", "Tatsa") for ll in plan["Llave"])), float((plan["Stock_TVL"] + plan["Stock_TATSA"]).sum())),
    ("Tránsito en PLAN = OC pendientes + RQ sin OC del saldo (unidades)", float(oc_pend.loc[oc_pend["Llave"].isin(plan["Llave"]), "Cant_Pendiente"].sum()
                                                                          + rq_soc.loc[rq_soc["Llave"].isin(plan["Llave"]), "Cant_Pendiente"].sum()),
     float((plan["RQ_COC_Und"] + plan["RQ_SOC_Und"]).sum())),
]
pd.Series(control_py).round(2)

##### 7. Excel: RESUMEN, PLAN y PROMETIDO

In [ ]:
F8, F8B, F8W = Font(name="Consolas", size=8), Font(name="Consolas", size=8, bold=True), Font(name="Consolas", size=8, bold=True, color="FFFFFF")
F_TIT, F_SUB = Font(name="Consolas", size=12, bold=True, color="1F4E78"), Font(name="Consolas", size=10, bold=True, color="1F4E78")
AMARILLO = PatternFill("solid", fgColor="FFF2CC")
COLOR = {"id": "1F4E78", "prom": "7030A0", "real": "BF8F00", "inv": "2E75B6", "precio": "375623", "calc": "C00000", "edit": "C65911", "abc": "595959", "trans": "0E7C7B"}
F_S, F_N, F_P = '"S/" #,##0;[Red]-"S/" #,##0;"-"', '#,##0.00;[Red]-#,##0.00;"-"', '0%;[Red]-0%;"-"'
F_S2 = '"S/" #,##0.00;[Red]-"S/" #,##0.00'
LINEA = Border(bottom=Side(style="thin", color="D9D9D9"))
PRM = {k: f"RESUMEN!${v[0]}${v[1:]}" for k, v in PARAM_CELDA.items()}
IGV_INV = '*IF({Compra_Con_IGV}="No",1,1+' + PRM["IGV"] + ')'
PY_CERR = "+".join("{PY_%s}" % MESES[m] for m in CERRADOS)
REAL_CERR = "+".join("{REAL_%s}" % MESES[m] for m in CERRADOS)


def pendiente(m):
    """Mes cerrado: 0 · mes en curso: lo que falta del plan · mes futuro: el plan."""
    M = MESES[m]
    return "=0" if m < MES_ACTUAL else (f"=MAX({{PLAN_{M}}}-{{REAL_{M}}},0)" if m == MES_ACTUAL else f"={{PLAN_{M}}}")


ACCION_F = ('=IF(AND({Compra_Und}>0.0001,{P_Com}=0),"ASIGNAR PRECIO",IF({Compra_Und}>0.0001,"COMPRAR",IF(OR(AND({Usa_COC}>0,{Dias_Atraso}>0),{Usa_SOC}>0),"APURAR TRÁNSITO",'
            'IF(AND({Inv_Final_Und}>0.0001,OR({Cobertura_Meses}>' + PRM["MESES_SOBRESTOCK"] + ',{Sobra_TVL_Und}>0.0001)),"ACELERAR CONSUMO",'
            'IF(AND(ABS({Desvio_Cerrado})>' + PRM["UMBRAL_DESVIO"] + ',ABS({Desvio_Cerrado_Soles})>' + PRM["MONTO_MIN_REVISAR"] + '),"REVISAR PROYECCIÓN","OK")))))')
# Bloques de PLAN en el orden de trabajo: (encabezado de la fila 1, color, columnas)
SECCIONES = [
    ("ACCIÓN", "calc", [("Accion", "calc", None, ACCION_F)]),
    ("1. REPUESTO · llave y códigos para comprar", "id", [
        ("Llave", "id", None, None), ("Producto", "id", None, None), ("Familia", "id", None, None),
        ("Cod_Ult_Compra", "id", None, None), ("Codigos_Llave", "id", None, None)]),
    ("2. CLASIFICACIÓN · método de proyección y ABC-XYZ del kardex", "abc", [
        ("Metodo", "abc", None, None), ("Cat_Prev", "abc", None, None), ("Cat_Corr", "abc", None, None), ("Proyecta", "abc", None, None),
        ("ABC_XYZ", "abc", None, None), ("Metodo_Sugerido", "abc", None, None), ("Revisar_Metodo", "abc", None, None)]),
    ("3. CONSUMO 12M · promedio y desviación mensual (und)", "real", [
        ("Prom_Mes_12M", "real", F_N, None), ("Desv_Mes_12M", "real", F_N, None), ("CV_12M", "real", F_N, None), ("Valor_Consumo_12M", "real", F_S, None)]),
    ("4. PRECIO · última compra real (IGV como Odoo)", "precio", [
        ("Ultima_OC", "precio", None, None), ("Fecha_Ult_Compra", "precio", None, None), ("Proveedor_Ult", "precio", None, None), ("Pais_Proveedor", "precio", None, None),
        ("Moneda_Ult", "precio", None, None), ("Importacion", "precio", None, None), ("Compra_Con_IGV", "precio", None, None),
        ("Costo_Inventario", "precio", F_S2, None), ("Precio_Compra", "precio", F_S2, None), ("PRECIO_MANUAL", "edit", F_S2, None),
        ("P_Inv", "precio", F_S2, "=IF(ISNUMBER({PRECIO_MANUAL}),{PRECIO_MANUAL},IF(ISNUMBER({Costo_Inventario}),{Costo_Inventario}" + IGV_INV
         + ",IF(ISNUMBER({Precio_Compra}),{Precio_Compra},0)))"),
        ("P_Com", "precio", F_S2, "=IF(ISNUMBER({PRECIO_MANUAL}),{PRECIO_MANUAL},IF(ISNUMBER({Precio_Compra}),{Precio_Compra},IF(ISNUMBER({Costo_Inventario}),{Costo_Inventario}"
         + IGV_INV + ",0)))")]),
    ("5. INVENTARIO HOY (und)", "inv", [("Stock_TVL", "inv", F_N, None), ("Stock_TATSA", "inv", F_N, None)]),
    ("6. EN TRÁNSITO · RQ con OC y sin OC", "trans", [
        ("RQ_COC_Und", "trans", F_N, None), ("OC_Pendiente", "trans", None, None), ("Comprador", "trans", None, None), ("Fecha_Prevista", "trans", None, None),
        ("Dias_Atraso", "trans", "#,##0", None), ("RQ_SOC_Und", "trans", F_N, None), ("RQ_Sin_OC", "trans", None, None)]),
    ("7. ANÁLISIS · lo prometido vs lo real (und)", "prom", [
        ("PY_OCT", "prom", F_N, None), ("REAL_OCT", "real", F_N, None), ("PY_NOV", "prom", F_N, None), ("REAL_NOV", "real", F_N, None),
        ("PY_DIC", "prom", F_N, None), ("REAL_DIC", "real", F_N, None),
        ("Desvio_Cerrado", "real", F_P, f"=IF(({PY_CERR})>0,({REAL_CERR})/({PY_CERR})-1,IF(({REAL_CERR})>0,1,0))"),
        ("Desvio_Cerrado_Soles", "real", F_S, f"=(({REAL_CERR})-({PY_CERR}))*{{P_Inv}}")]),
    ("8. ANÁLISIS · ajuste manual y pendiente (und)", "edit", [
        ("AJ_NOV", "edit", F_N, None), ("AJ_DIC", "edit", F_N, None),
        ("PLAN_NOV", "prom", F_N, "=IF(ISNUMBER({AJ_NOV}),{AJ_NOV},{PY_NOV})"), ("PLAN_DIC", "prom", F_N, "=IF(ISNUMBER({AJ_DIC}),{AJ_DIC},{PY_DIC})"),
        ("Pend_NOV", "prom", F_N, pendiente(11)), ("Pend_DIC", "prom", F_N, pendiente(12))]),
    ("9. ANÁLISIS · cobertura (TVL → TATSA → tránsito) y compra", "calc", [
        ("Usa_TVL", "calc", F_N, "=MIN({Stock_TVL},{Pend_NOV}+{Pend_DIC})"), ("Usa_TATSA", "calc", F_N, "=MIN({Stock_TATSA},{Pend_NOV}+{Pend_DIC}-{Usa_TVL})"),
        ("Usa_COC", "calc", F_N, "=MIN({RQ_COC_Und},{Pend_NOV}+{Pend_DIC}-{Usa_TVL}-{Usa_TATSA})"),
        ("Usa_SOC", "calc", F_N, "=MIN({RQ_SOC_Und},{Pend_NOV}+{Pend_DIC}-{Usa_TVL}-{Usa_TATSA}-{Usa_COC})"),
        ("Compra_NOV_Und", "calc", F_N, "=MAX({Pend_NOV}-({Stock_TVL}+{Stock_TATSA}+{RQ_COC_Und}+{RQ_SOC_Und}),0)"),
        ("Compra_DIC_Und", "calc", F_N, "={Compra_Und}-{Compra_NOV_Und}"),
        ("Compra_Und", "calc", F_N, "={Pend_NOV}+{Pend_DIC}-{Usa_TVL}-{Usa_TATSA}-{Usa_COC}-{Usa_SOC}"),
        ("Compra_NOV_Soles", "calc", F_S, "={Compra_NOV_Und}*{P_Com}"), ("Compra_DIC_Soles", "calc", F_S, "={Compra_DIC_Und}*{P_Com}"),
        ("Compra_Soles", "calc", F_S, "={Compra_Und}*{P_Com}"), ("Usa_Almacen_Soles", "calc", F_S, "=({Usa_TVL}+{Usa_TATSA})*{P_Inv}"),
        ("Usa_Transito_Soles", "calc", F_S, "=({Usa_COC}+{Usa_SOC})*{P_Com}")]),
    # Inventario al cierre de cada mes: TVL se consume primero, luego TATSA y al final el tránsito (que llega a TATSA)
    ("10. ANÁLISIS · inventario proyectado al cierre de NOV y DIC", "inv", [
        ("Inv_TVL_NOV", "inv", F_N, "=MAX({Stock_TVL}-{Pend_NOV},0)"),
        ("Inv_TATSA_NOV", "inv", F_N, "=MAX({Stock_TATSA}-MAX({Pend_NOV}-{Stock_TVL},0),0)"),
        ("Trans_NOV", "inv", F_N, "=MAX({RQ_COC_Und}+{RQ_SOC_Und}-MAX({Pend_NOV}-{Stock_TVL}-{Stock_TATSA},0),0)"),
        ("Inv_TVL_DIC", "inv", F_N, "={Stock_TVL}-{Usa_TVL}"), ("Inv_TATSA_DIC", "inv", F_N, "={Stock_TATSA}-{Usa_TATSA}"),
        ("Trans_DIC", "inv", F_N, "={RQ_COC_Und}+{RQ_SOC_Und}-{Usa_COC}-{Usa_SOC}"),
        ("Inv_Final_Und", "inv", F_N, "={Inv_TVL_DIC}+{Inv_TATSA_DIC}+{Trans_DIC}"),
        ("Inv_Final_Soles", "inv", F_S, "=({Inv_TVL_DIC}+{Inv_TATSA_DIC})*{P_Inv}+{Trans_DIC}*{P_Com}"), ("Sobra_TVL_Und", "inv", F_N, "={Inv_TVL_DIC}"),
        ("Cobertura_Meses", "inv", F_N, "=IF({Prom_Mes_12M}>0,{Inv_Final_Und}/{Prom_Mes_12M},IF({Inv_Final_Und}>0,999,0))")]),
    ("NOTAS", "edit", [("COMENTARIO", "edit", None, None)]),
    ("DETALLE (plegado: pulsar + sobre la columna)", "abc", [
        ("PY_OCT_PREV", "prom", F_N, None), ("PY_OCT_CORR", "prom", F_N, None), ("REAL_OCT_PREV", "real", F_N, None), ("REAL_OCT_CORR", "real", F_N, None),
        ("PY_NOV_PREV", "prom", F_N, None), ("PY_NOV_CORR", "prom", F_N, None), ("REAL_NOV_PREV", "real", F_N, None), ("REAL_NOV_CORR", "real", F_N, None),
        ("PY_DIC_PREV", "prom", F_N, None), ("PY_DIC_CORR", "prom", F_N, None), ("REAL_DIC_PREV", "real", F_N, None), ("REAL_DIC_CORR", "real", F_N, None)]
     + [(n, "inv", F_N, None) for n in STOCK_FIN]
     + [("Tipo", "id", None, None), ("Llave_13OCT", "id", None, None), ("Nota_Llave", "id", None, None), ("Origen", "id", None, None)]),
]
COLS = [col for _, _, cols in SECCIONES for col in cols]
NOMBRES = [x[0] for x in COLS]
L = {n: get_column_letter(i) for i, n in enumerate(NOMBRES, start=1)}
orden = {c_: i for i, c_ in enumerate(CATEGORIAS)}
plan = plan.assign(_o=plan["Metodo"].map(orden), _v=(plan[["PY_NOV", "PY_DIC"]].sum(axis=1) + plan["REAL_OCT"]) * plan["Precio_Compra"].fillna(0)).sort_values(
    ["_o", "Familia", "_v"], ascending=[True, True, False]).drop(columns=["_o", "_v"]).reset_index(drop=True)
FILA_1 = 3                                   # fila 1: bloques · fila 2: columnas · datos desde la fila 3
N, ULT = len(plan), len(plan) + FILA_1 - 1


def formula(tpl, fila):
    return tpl.format(**{n: f"{L[n]}{fila}" for n in NOMBRES})


def rango(nombre):
    return f"PLAN!${L[nombre]}${FILA_1}:${L[nombre]}${ULT}"


def encabezado(hoja, fila, columnas, grupos, col0=1):
    for i, (nom, g) in enumerate(zip(columnas, grupos), start=col0):
        if nom == "":
            continue
        x = hoja.cell(fila, i, nom)
        x.font, x.fill = F8W, PatternFill("solid", fgColor=COLOR[g])
        x.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    hoja.row_dimensions[fila].height = 28


wb = Workbook()
hr = wb.active
hr.title = "RESUMEN"
pl = wb.create_sheet("PLAN")
hp = wb.create_sheet("PROMETIDO")

# ---------- PLAN ----------
col0 = 1
for texto, color, cols in SECCIONES:                       # fila 1: qué es cada bloque
    col1 = col0 + len(cols) - 1
    if col1 > col0:
        pl.merge_cells(start_row=1, start_column=col0, end_row=1, end_column=col1)
    x = pl.cell(1, col0, texto)
    x.font, x.alignment = F8W, Alignment(horizontal="left", vertical="center", wrap_text=True, indent=1)
    for k in range(col0, col1 + 1):
        pl.cell(1, k).fill = PatternFill("solid", fgColor=COLOR[color])
        pl.cell(1, k).border = Border(left=Side(style="medium", color="FFFFFF") if k == col0 else None)
    col0 = col1 + 1
pl.row_dimensions[1].height = 30
encabezado(pl, 2, NOMBRES, [x[1] for x in COLS])
for r, fila in enumerate(plan.itertuples(index=False), start=FILA_1):
    datos = fila._asdict()
    for i, (nom, g, fmt, tpl) in enumerate(COLS, start=1):
        v = formula(tpl, r) if tpl else datos.get(nom)
        if isinstance(v, float) and np.isnan(v):
            v = None
        x = pl.cell(r, i, v)
        x.font, x.border = F8, LINEA
        if fmt:
            x.number_format = fmt
        x.alignment = Alignment(horizontal="right" if fmt else "left", vertical="center")
        if g == "edit":
            x.fill = AMARILLO
for i, nom in enumerate(NOMBRES, start=1):
    pl.column_dimensions[get_column_letter(i)].width = {"Accion": 19, "Llave": 11, "Producto": 34, "Familia": 15, "Metodo": 16, "Cat_Prev": 14, "Cat_Corr": 18,
                                                        "Proyecta": 16, "Metodo_Sugerido": 16, "OC_Pendiente": 16, "Comprador": 18, "RQ_Sin_OC": 14,
                                                        "Proveedor_Ult": 24, "COMENTARIO": 30, "Tipo": 24, "Nota_Llave": 30, "Origen": 30,
                                                        "Cod_Ult_Compra": 12, "Codigos_Llave": 28, "Ultima_OC": 14}.get(nom, 11)
detalle_cols = SECCIONES[-1][2]
pl.column_dimensions.group(L[detalle_cols[0][0]], L[detalle_cols[-1][0]], outline_level=1, hidden=True)
pl.freeze_panes = f"D{FILA_1}"
pl.auto_filter.ref = f"A2:{L[NOMBRES[-1]]}{ULT}"
pl.sheet_view.showGridLines = False
for texto, color in [("COMPRAR", "C00000"), ("ASIGNAR PRECIO", "7B1E1E"), ("APURAR TRÁNSITO", "C65911"), ("ACELERAR CONSUMO", "2E75B6"), ("REVISAR PROYECCIÓN", "7030A0")]:
    pl.conditional_formatting.add(f"A{FILA_1}:A{ULT}", CellIsRule(operator="equal", formula=[f'"{texto}"'], font=Font(name="Consolas", size=8, bold=True, color=color)))
pl.conditional_formatting.add(f"{L['Revisar_Metodo']}{FILA_1}:{L['Revisar_Metodo']}{ULT}", CellIsRule(operator="equal", formula=['"Sí"'], font=Font(name="Consolas", size=8, bold=True, color="C00000")))
for col in ["AJ_NOV", "AJ_DIC", "PRECIO_MANUAL"]:
    dv = DataValidation(type="decimal", operator="greaterThanOrEqual", formula1="0", allow_blank=True, error="Solo números ≥ 0 (vacío = usar el cálculo)")
    pl.add_data_validation(dv)
    dv.add(f"{L[col]}{FILA_1}:{L[col]}{ULT}")

# ---------- PROMETIDO ----------
pr = proy.assign(Familia=proy["Llave"].map(FAMILIA))[["Llave_13OCT", "Llave", "Producto", "Familia", "Tipo", "Categoria", "OCT", "NOV", "DIC", "Cuenta_Como_Consumo",
                                                     "Nota_Llave"]].rename(columns={"Llave": "Llave_Actual"})
encabezado(hp, 1, list(pr.columns), ["id"] * 6 + ["prom"] * 3 + ["id"] * 2)
for r, fila in enumerate(pr.itertuples(index=False), start=2):
    for i, v in enumerate(fila, start=1):
        x = hp.cell(r, i, v)
        x.font, x.border = F8, LINEA
        if 7 <= i <= 9:
            x.number_format = F_N
for i, w in enumerate([12, 12, 34, 15, 24, 20, 10, 10, 10, 12, 40], start=1):
    hp.column_dimensions[get_column_letter(i)].width = w
hp.freeze_panes = "C2"; hp.auto_filter.ref = hp.dimensions; hp.sheet_view.showGridLines = False

# ---------- RESUMEN ----------
from openpyxl.chart.label import DataLabelList
from openpyxl.chart.axis import ChartLines
from openpyxl.chart.shapes import GraphicalProperties
from openpyxl.drawing.line import LineProperties

AZUL_DATO = "0000FF"                       # dato de origen (no es fórmula)
GRIS_BORDE = Side(style="thin", color="BFBFBF")
BORDE = Border(left=GRIS_BORDE, right=GRIS_BORDE, top=GRIS_BORDE, bottom=GRIS_BORDE)
BORDE_TOTAL = Border(left=GRIS_BORDE, right=GRIS_BORDE, top=Side(style="medium", color="404040"), bottom=GRIS_BORDE)
FONDO_TOTAL = PatternFill("solid", fgColor="F2F2F2")
F_DESV = '+0%;[Red]-0%;"-"'
PALETA = ["1F4E78", "C55A11", "7F7F7F", "BF8F00", "2E75B6", "A5A5A5"]
hr.sheet_view.showGridLines = False
for col, w in zip("ABCDEFGHIJK", [58, 16, 16, 16, 16, 16, 14, 3, 3, 30, 11]):
    hr.column_dimensions[col].width = w
hr.column_dimensions["M"].width, hr.column_dimensions["N"].width = 22, 16


def celda(ref, valor, fmt=None, negrita=False, dato=False, alinear=None, borde=None, fondo=None):
    x = hr[ref]
    x.value = valor
    x.font = Font(name="Consolas", size=8, bold=negrita, color=AZUL_DATO if dato else None)
    if fmt:
        x.number_format = fmt
    x.alignment = Alignment(horizontal=alinear or ("left" if ref[0] == "A" or isinstance(valor, str) and not str(valor).startswith("=") else "right"),
                            vertical="center", wrap_text=False)
    if borde:
        x.border = borde
    if fondo:
        x.fill = fondo
    return x


def titulo(fila, texto):
    x = hr.cell(fila, 1, texto)
    x.font = F_SUB


def cabecera(fila, textos, colores, col0=1):
    for i, (t, c_) in enumerate(zip(textos, colores), start=col0):
        x = hr.cell(fila, i, t)
        x.font, x.fill, x.border = F8W, PatternFill("solid", fgColor=COLOR[c_]), BORDE
        x.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    hr.row_dimensions[fila].height = 30


def fila(n, etiqueta, valores, fmt=F_S, total=False, datos=(), ncols=None):
    """valores: lista para columnas B, C, ...; datos: índices (0 = B) que son datos de origen (azul)."""
    ncols = ncols or len(valores)
    b = BORDE_TOTAL if total else BORDE
    f = FONDO_TOTAL if total else None
    celda(f"A{n}", etiqueta, negrita=total, borde=b, fondo=f)
    for i in range(ncols):
        v = valores[i] if i < len(valores) else None
        celda(f"{get_column_letter(i + 2)}{n}", v, fmt, negrita=total, dato=i in datos, alinear="right", borde=b, fondo=f)


def estilo(g, alto=7.5, ancho=12.5, colores=PALETA, etiquetas=False, fmt_eje='#,##0,"k"'):
    g.height, g.width = alto, ancho
    g.legend.position = "b"
    g.legend.overlay = False
    if hasattr(g, "y_axis"):
        g.x_axis.delete = False
        g.y_axis.delete = False
        g.y_axis.numFmt = fmt_eje
        g.y_axis.majorGridlines = ChartLines(spPr=GraphicalProperties(ln=LineProperties(solidFill="E7E6E6")))
    for i, serie in enumerate(g.series):
        color = colores[i % len(colores)]
        if isinstance(g, LineChart):
            serie.graphicalProperties.line.solidFill = color
            serie.graphicalProperties.line.width = 28000
            serie.marker.symbol, serie.marker.size = "circle", 7
            serie.marker.graphicalProperties = GraphicalProperties(solidFill=color, ln=LineProperties(solidFill=color))
            serie.smooth = False
        else:
            serie.graphicalProperties.solidFill = color
            serie.graphicalProperties.line.solidFill = color
        if etiquetas:
            serie.dLbls = DataLabelList()
            serie.dLbls.showVal, serie.dLbls.numFmt = True, '#,##0,"k"'
            for k in ("showSerName", "showCatName", "showLegendKey", "showPercent"):
                setattr(serie.dLbls, k, False)


def nombres(g, *textos):
    for serie, t in zip(g.series, textos):
        serie.tx = SeriesLabel(v=t)


# Encabezado
celda("A1", f"TATSA · [{CENTRO}] · {SITUACION} · cierre {ANIO}").font = F_TIT
celda("A2", f"Datos al {HOY:%d/%m/%Y} · mes en curso: {NOMBRE_MES[MES_ACTUAL].lower()} · meses cerrados: {TXT_CERRADOS} (solo lo real) · punto de partida 01/10/{ANIO}")
celda("A3", f"Azul = dato de origen · negro = fórmula · amarillo = editable · Fuente: {ARCHIVO_SALDO} ({FECHA_SALDO}) y kardex al {kardex['Fecha'].max():%d/%m/%Y}")
hr["A3"].font = Font(name="Consolas", size=8, italic=True, color="595959")
GRIS = Font(name="Consolas", size=8, italic=True, color="595959")
NOMBRE_COL = {m: get_column_letter(i) for i, m in enumerate(MESES, start=3)}          # OCT = C, NOV = D, DIC = E

# Parámetros
celda("J4", "Parámetros").font = F_SUB
for k, desc in [("IGV", "IGV (compras con IGV)"), ("MESES_SOBRESTOCK", "Meses de stock = sobrestock"), ("UMBRAL_DESVIO", "Desvío para revisar (%)"),
                ("MONTO_MIN_REVISAR", "… y monto mínimo (S/)")]:
    ref = PARAM_CELDA[k]
    celda(f"J{ref[1:]}", desc, borde=BORDE)
    celda(ref, parametros[k], F_P if k in ("IGV", "UMBRAL_DESVIO") else "#,##0", negrita=True, alinear="right", borde=BORDE, fondo=AMARILLO)

# ===== 1. Presupuesto: cómo se ha consumido =====
S1 = 10
titulo(S1, f"1. Presupuesto {ANIO}: cómo se ha consumido hasta hoy")
cabecera(S1 + 1, ["Concepto", "S/", "% del presupuesto"], ["id", "id", "id"])
PPTO, ENESET = S1 + 2, S1 + 3
filas_ppto = [(PPTO, "Presupuesto vigente", SALDO["Presupuesto"]), (ENESET, "(−) Compras ejecutadas enero – setiembre", SALDO["Ejec_ENE_SET"])]
FILA_EJEC = {}
for m in MESES:
    if m <= MES_ACTUAL:
        FILA_EJEC[m] = PPTO + len(filas_ppto)
        filas_ppto.append((FILA_EJEC[m], f"(−) Compras ejecutadas {NOMBRE_MES[m].lower()}" + (" (a la fecha)" if m == MES_ACTUAL else ""), EJEC_MES[m]))
COC, SOC = PPTO + len(filas_ppto), PPTO + len(filas_ppto) + 1
filas_ppto += [(COC, "(−) En tránsito con OC (pendiente de entrega)", SALDO["Transito_COC"]), (SOC, "(−) RQ aprobados sin OC", SALDO["Transito_SOC"])]
for n, t, v in filas_ppto:
    fila(n, t, [round(v, 2), f"=IF($B${PPTO}=0,0,B{n}/$B${PPTO})"], datos=(0,))
    hr[f"C{n}"].number_format = "0.0%"
HOYF = SOC + 1
fila(HOYF, "→ Saldo real hoy", [f"=B{PPTO}-SUM(B{ENESET}:B{SOC})", f"=IF($B${PPTO}=0,0,B{HOYF}/$B${PPTO})"], total=True)
hr[f"C{HOYF}"].number_format = "0.0%"
celda(f"A{HOYF + 1}", f"Odoo muestra como disponible S/ {SALDO['Saldo_Odoo']:,.0f}: no descuenta el tránsito ni las importaciones facturadas en otro año.").font = GRIS
celda(f"A{HOYF + 2}", "Compras y tránsito: monto real con el que se descuenta el presupuesto (Excel del saldo).").font = GRIS

# Datos de los gráficos (columnas M:N)
celda(f"M{S1}", "Datos para gráficos").font = F8B
cabecera(S1 + 1, ["Mes", "Compras ejecutadas"], ["id", "id"], col0=13)
meses_g = [m for m in range(1, 13) if m <= MES_ACTUAL]
ABREV = ["ENE", "FEB", "MAR", "ABR", "MAY", "JUN", "JUL", "AGO", "SET", "OCT", "NOV", "DIC"]
for i, m in enumerate(meses_g, start=S1 + 2):
    celda(f"M{i}", ABREV[m - 1], borde=BORDE, alinear="center")
    valor_m = f"=B{FILA_EJEC[m]}" if m in FILA_EJEC else round(float(EJECUTADO[m]), 2)
    celda(f"N{i}", valor_m, F_S, dato=m not in FILA_EJEC, borde=BORDE)
FIN_G = S1 + 1 + len(meses_g)
r0 = FIN_G + 2
cabecera(r0, ["Destino del presupuesto", "S/"], ["id", "id"], col0=13)
destino = [("Compras ene–set", ENESET)] + [(f"Compras {NOMBRE_MES[m].lower()}", FILA_EJEC[m]) for m in FILA_EJEC] + [("Tránsito con OC", COC), ("RQ sin OC", SOC), ("Saldo hoy", HOYF)]
for i, (t, ref) in enumerate(destino, start=r0 + 1):
    celda(f"M{i}", t, borde=BORDE)
    celda(f"N{i}", f"=MAX(B{ref},0)", F_S, borde=BORDE)
celda(f"M{r0 + len(destino) + 1}", "Presupuesto", borde=BORDE)       # categoría única de la barra apilada

GP = HOYF + 4
g = BarChart(); g.type, g.grouping, g.overlap, g.title = "bar", "percentStacked", 100, "¿En qué se fue el presupuesto? (% y S/)"
g.add_data(Reference(hr, min_col=14, min_row=r0 + 1, max_row=r0 + len(destino)), from_rows=True, titles_from_data=False)
g.set_categories(Reference(hr, min_col=13, min_row=r0 + len(destino) + 1))
nombres(g, *[t for t, _ in destino])
estilo(g, alto=6.5, ancho=11.5, colores=["1F4E78", "2E75B6", "9DC3E6", "C55A11", "F4B183", "70AD47"][-len(destino):] if len(destino) <= 6
       else ["1F4E78", "2E75B6", "5B9BD5", "9DC3E6", "C55A11", "F4B183", "70AD47"], etiquetas=True, fmt_eje="0%")
g.gapWidth = 40
g.x_axis.delete = True
for serie in g.series[1:]:                       # tramos chicos: sin etiqueta (se leen en la leyenda y en la tabla)
    serie.dLbls = None
hr.add_chart(g, f"A{GP}")
g = BarChart(); g.type, g.title = "col", "Compras ejecutadas por mes (S/)"
g.add_data(Reference(hr, min_col=14, min_row=S1 + 2, max_row=FIN_G), titles_from_data=False)
g.set_categories(Reference(hr, min_col=13, min_row=S1 + 2, max_row=FIN_G))
nombres(g, "Compras ejecutadas")
estilo(g, alto=6.5, ancho=14, colores=["1F4E78"])
g.legend = None
hr.add_chart(g, f"B{GP}")

# ===== 2. Mes a mes =====
R = GP + 15
titulo(R, "2. Mes a mes desde el 01/10: lo prometido, lo consumido, lo comprado, el saldo y el inventario")
cabecera(R + 1, ["S/", "Inicio (30/09)"] + [f"{NOMBRE_MES[m]} ({ESTADO[m]})" for m in MESES] + ["Total OCT–DIC"],
         ["id", "id"] + ["real" if m < MES_ACTUAL else "prom" for m in MESES] + ["id"])
sp = lambda campo, precio="P_Inv": f"=SUMPRODUCT({rango(campo)},{rango(precio)})"
r = {k: R + 2 + i for i, k in enumerate(["prom", "cons", "dif", "ejec", "trans", "falta", "gasto", "saldo", "tvl", "tatsa", "transinv", "inv"])}
tot = lambda k: f"=SUM(C{r[k]}:E{r[k]})"
fila(r["prom"], "Prometido de consumo (archivo del 13/10)", [None] + [sp(f"PY_{MESES[m]}") for m in MESES] + [tot("prom")])
fila(r["cons"], "Consumo: real (cerrado) · real + falta (en curso) · plan", [None] + [
    sp(f"REAL_{MESES[m]}") if m < MES_ACTUAL else f"=SUMPRODUCT({rango('REAL_' + MESES[m])}+{rango('Pend_' + MESES[m])},{rango('P_Inv')})" for m in MESES] + [tot("cons")])
fila(r["dif"], "Diferencia consumo − prometido", [None] + [f"={c_}{r['cons']}-{c_}{r['prom']}" for c_ in "CDEF"])
fila(r["ejec"], "Compras ejecutadas (recibidas o facturadas)", [None] + [f"=B{FILA_EJEC[m]}" if m in FILA_EJEC else 0 for m in MESES] + [tot("ejec")])
fila(r["trans"], "Tránsito comprometido (RQ con OC + sin OC: va al mes en curso)", [None] + [f"=B{COC}+B{SOC}" if m == MES_ACTUAL else 0 for m in MESES] + [tot("trans")])
fila(r["falta"], "Compra que falta (proyección)", [None] + [f"=SUM({rango('Compra_' + MESES[m] + '_Soles')})" if m >= MES_ACTUAL else 0 for m in MESES] + [tot("falta")])
fila(r["gasto"], "Gasto de presupuesto del mes", [None] + [f"=SUM({c_}{r['ejec']}:{c_}{r['falta']})" for c_ in "CDEF"], total=True)
fila(r["saldo"], "Saldo disponible al cierre del mes", [f"=B{PPTO}-B{ENESET}", f"=B{r['saldo']}-C{r['gasto']}", f"=C{r['saldo']}-D{r['gasto']}", f"=D{r['saldo']}-E{r['gasto']}", None], total=True)


def inv_mes(m, col):
    """Inventario al cierre: real del kardex en meses cerrados; proyectado en el mes en curso y los siguientes."""
    return sp(f"Stock_{col}_{FIN_MES[m]}") if m < MES_ACTUAL else sp(f"Inv_{col}_{MESES[m]}")


fila(r["tvl"], "Inventario almacén TVL", [sp("Stock_TVL_30SET")] + [inv_mes(m, "TVL") for m in MESES] + [None])
fila(r["tatsa"], "Inventario almacén TATSA", [sp("Stock_TATSA_30SET")] + [inv_mes(m, "TATSA") for m in MESES] + [None])
fila(r["transinv"], "Tránsito por recibir / sin consumir (llega a TATSA)", [None] + [None if m < MES_ACTUAL else sp(f"Trans_{MESES[m]}", "P_Com") for m in MESES] + [None])
fila(r["inv"], "Inventario total al cierre del mes", [f"=SUM({c_}{r['tvl']}:{c_}{r['transinv']})" for c_ in "BCDE"] + [None], total=True)
for c_ in "CDEF":
    hr.conditional_formatting.add(f"{c_}{r['dif']}", CellIsRule(operator="greaterThan", formula=["0"], font=Font(name="Consolas", size=8, bold=True, color="C00000")))
hr.conditional_formatting.add(f"B{r['saldo']}:E{r['saldo']}", CellIsRule(operator="lessThan", formula=["0"], font=Font(name="Consolas", size=8, bold=True, color="C00000"),
                                                                       fill=PatternFill("solid", fgColor="FBEDEC")))
notas = [f"Saldo al 30/09 = presupuesto − compras ene–set, sin descontar el tránsito. Con los RQ de ene–set aún en tránsito sería S/ {SALDO['Saldo_30SET_Con_Transito']:,.0f}: "
         f"ese tránsito se cuenta en {NOMBRE_MES[MES_ACTUAL].lower()}.",
         "Mes cerrado = solo lo comprado. El tránsito de cualquier mes va al mes en curso. Consumo e inventario: al precio vigente del repuesto; "
         "compras y tránsito: al monto real.",
         "Inventario: TVL se consume primero, luego TATSA y al final el tránsito. Meses cerrados: stock real del kardex. El tránsito no consumido llega a TATSA."]
for i, t in enumerate(notas, start=r["inv"] + 1):
    celda(f"A{i}", t).font = GRIS
# Etiquetas cortas para los gráficos (columnas M..P)
for j, t in enumerate(["30/09"] + [NOMBRE_MES[m] for m in MESES]):
    celda(f"{get_column_letter(13 + j)}{R + 1}", t, alinear="center")
CAT4 = Reference(hr, min_col=13, max_col=16, min_row=R + 1)
CAT3 = Reference(hr, min_col=14, max_col=16, min_row=R + 1)

G = r["inv"] + len(notas) + 2
g = BarChart(); g.type, g.grouping, g.title = "col", "clustered", "Consumo: prometido vs real / plan (S/)"
g.add_data(Reference(hr, min_col=3, max_col=5, min_row=r["prom"], max_row=r["cons"]), from_rows=True, titles_from_data=False)
g.set_categories(CAT3)
nombres(g, "Prometido", "Real / plan"); estilo(g, ancho=11.5, colores=["A5A5A5", "1F4E78"])
hr.add_chart(g, f"A{G}")
g = BarChart(); g.type, g.grouping, g.overlap, g.title = "col", "stacked", 100, "Gasto de presupuesto por mes (S/)"
g.add_data(Reference(hr, min_col=3, max_col=5, min_row=r["ejec"], max_row=r["falta"]), from_rows=True, titles_from_data=False)
g.set_categories(CAT3)
nombres(g, "Compras ejecutadas", "Tránsito", "Falta comprar"); estilo(g, ancho=14, colores=["1F4E78", "C55A11", "F4B183"])
hr.add_chart(g, f"B{G}")
G2 = G + 16
g = LineChart(); g.title = "Saldo disponible al cierre del mes (S/)"
g.add_data(Reference(hr, min_col=2, max_col=5, min_row=r["saldo"]), from_rows=True, titles_from_data=False)
g.set_categories(CAT4)
nombres(g, "Saldo disponible"); estilo(g, ancho=11.5, colores=["1F4E78"]); g.legend = None
hr.add_chart(g, f"A{G2}")
g = BarChart(); g.type, g.grouping, g.overlap, g.title = "col", "stacked", 100, "Inventario al cierre del mes: TVL, TATSA y tránsito (S/)"
g.add_data(Reference(hr, min_col=2, max_col=5, min_row=r["tvl"], max_row=r["transinv"]), from_rows=True, titles_from_data=False)
g.set_categories(CAT4)
nombres(g, "Almacén TVL", "Almacén TATSA", "Tránsito por recibir"); estilo(g, ancho=14, colores=["BF8F00", "2E75B6", "9DC3E6"])
hr.add_chart(g, f"B{G2}")

# ===== 3. Qué hay que hacer =====
A3 = G2 + 17
titulo(A3, "3. ¿Qué hay que hacer?   (en PLAN, filtrar la columna Accion)")
cabecera(A3 + 1, ["Acción", "Repuestos", "S/", "Qué significa"], ["calc", "calc", "calc", "calc"])
AJ_TXT = "AJ_NOV / AJ_DIC" if MES_ACTUAL == 11 else "AJ_DIC"
acciones = [("COMPRAR", "Compra_Soles", "No alcanza con stock TVL + TATSA + tránsito (S/ = compra que falta)"),
            ("APURAR TRÁNSITO", "Usa_Transito_Soles", "Se necesita lo que está en tránsito: OC atrasada o RQ sin OC"),
            ("ACELERAR CONSUMO", "Inv_Final_Soles", "Sobra stock al 31-DIC o en TVL: consumir primero, no comprar"),
            ("REVISAR PROYECCIÓN", None, f"El real de {TXT_CERRADOS} se desvió de lo prometido: ajustar {AJ_TXT}"),
            ("ASIGNAR PRECIO", None, "Hay que comprar pero no tiene precio: escribir PRECIO_MANUAL")]
FILA_ACC = {}
for i, (acc, col_s, desc) in enumerate(acciones, start=A3 + 2):
    FILA_ACC[acc] = i
    celda(f"A{i}", acc, negrita=True, borde=BORDE)
    celda(f"B{i}", f"=COUNTIF({rango("Accion")},A{i})", "#,##0", borde=BORDE)
    celda(f"C{i}", f"=SUMIF({rango("Accion")},A{i},{rango(col_s)})" if col_s else None, F_S, borde=BORDE)
    hr.merge_cells(f"D{i}:F{i}")
    celda(f"D{i}", desc, borde=BORDE, alinear="left")
    hr[f"D{i}"].alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
    for c_ in "EF":
        hr[f"{c_}{i}"].border = BORDE
    hr.row_dimensions[i].height = 24

# ===== 4. Cumplimiento por categoría =====
C4 = A3 + 9
MC = MESES[MES_ACTUAL]
titulo(C4, "4. ¿Se cumplió lo prometido? Por categoría (el consumo real se separa por tipo de OT: preventiva / correctiva)")
cabecera(C4 + 1, ["Categoría", f"Prometido {TXT_CERRADOS}", f"Real {TXT_CERRADOS}", "Desvío", f"Prometido {MC}", f"Real {MC} a hoy"], ["id", "prom", "real", "real", "prom", "real"])


def por_categoria(cat, campos):
    partes = []
    for campo in campos:
        if cat in ("PREVENTIVO", "NO PROYECTADO"):
            partes.append(f'SUMPRODUCT(({rango("Cat_Prev")}="{cat}")*{rango(campo + "_PREV")}*{rango("P_Inv")})')
        if cat != "PREVENTIVO":
            partes.append(f'SUMPRODUCT(({rango("Cat_Corr")}="{cat}")*{rango(campo + "_CORR")}*{rango("P_Inv")})')
    return "=" + "+".join(partes)


grupos = [[f"PY_{MESES[m]}" for m in CERRADOS], [f"REAL_{MESES[m]}" for m in CERRADOS], None, [f"PY_{MC}"], [f"REAL_{MC}"]]
for i, cat in enumerate(CATEGORIAS + ["TOTAL"], start=C4 + 2):
    total = cat == "TOTAL"
    vals = []
    for j, campos in enumerate(grupos):
        col = get_column_letter(j + 2)
        if campos is None:
            vals.append(f'=IF(B{i}>0,C{i}/B{i}-1,"-")')
        elif total:
            vals.append(f"=SUM({col}{C4 + 2}:{col}{i - 1})")
        else:
            vals.append(por_categoria(cat, campos))
    fila(i, cat.capitalize() if not total else "TOTAL", vals, total=total)
    hr[f"D{i}"].number_format = F_DESV
FIN_CAT = C4 + 2 + len(CATEGORIAS)
hr.conditional_formatting.add(f"D{C4 + 2}:D{FIN_CAT}", FormulaRule(formula=[f"AND(ISNUMBER(D{C4 + 2}),ABS(D{C4 + 2})>{PRM['UMBRAL_DESVIO']})"],
                                                                   font=Font(name="Consolas", size=8, bold=True, color="C00000")))
g = BarChart(); g.type, g.grouping, g.title = "bar", "clustered", f"{TXT_CERRADOS} por categoría: prometido vs real (S/)"
g.add_data(Reference(hr, min_col=2, max_col=3, min_row=C4 + 1, max_row=FIN_CAT - 1), titles_from_data=True)
g.set_categories(Reference(hr, min_col=1, min_row=C4 + 2, max_row=FIN_CAT - 1))
estilo(g, alto=8, ancho=17, colores=["A5A5A5", "1F4E78"], etiquetas=True)
g.x_axis.scaling.orientation, g.y_axis.crosses, g.legend.position = "maxMin", "max", "r"
hr.add_chart(g, f"A{FIN_CAT + 2}")

# ===== 5. Controles (compactos y plegados) =====
K = FIN_CAT + 19
sp_cerr = lambda pre: "=" + "+".join(sp(f"{pre}_{MESES[m]}")[1:] for m in CERRADOS)
controles = [(t, round(e, 2), round(o, 2), True) for t, e, o in CONTROLES_PY] + [
    ("Saldo hoy de la sección 1 = saldo real del Excel del saldo", round(SALDO["Saldo_Hoy"], 2), f"=B{HOYF}", False),
    (f"Consumo real {TXT_CERRADOS}: preventivo + correctivo = total (und)", "=" + "+".join(f"SUM({rango('REAL_' + MESES[m])})" for m in CERRADOS),
     "=" + "+".join(f"SUM({rango('REAL_' + MESES[m] + '_PREV')})+SUM({rango('REAL_' + MESES[m] + '_CORR')})" for m in CERRADOS), False),
    (f"Cumplimiento: real {TXT_CERRADOS} por categoría = total (S/)", sp_cerr("REAL"), f"=C{FIN_CAT}", False),
    (f"Cumplimiento: prometido {TXT_CERRADOS} por categoría = total (S/)", sp_cerr("PY"), f"=B{FIN_CAT}", False),
    ("Compra de noviembre + diciembre = compra total (S/)", f"=SUM({rango('Compra_Soles')})", f"=SUM({rango('Compra_NOV_Soles')})+SUM({rango('Compra_DIC_Soles')})", False),
    ("Saldo al 31-DIC = saldo hoy − compra que falta", f"=B{HOYF}-SUM({rango('Compra_Soles')})", f"=E{r['saldo']}", False),
    ("Inventario al 31-DIC: TVL + TATSA + tránsito = total de PLAN (S/)", f"=SUM({rango('Inv_Final_Soles')})", f"=E{r['inv']}", False),
    ("Unidades: lo pendiente se cubre con stock + tránsito + compra (und)", f"=SUM({rango('Pend_NOV')})+SUM({rango('Pend_DIC')})",
     f"=SUM({rango('Usa_TVL')})+SUM({rango('Usa_TATSA')})+SUM({rango('Usa_COC')})+SUM({rango('Usa_SOC')})+SUM({rango('Compra_Und')})", False),
]
NK = len(controles)
titulo(K, "5. Controles")
celda(f"B{K}", f'=COUNTIF(D{K + 2}:D{K + 1 + NK},"OK")&" de {NK} OK"', negrita=True, alinear="left")
celda(f"C{K}", "(pulsar + a la izquierda para ver el detalle)").font = GRIS
cabecera(K + 1, ["Control", "Esperado", "Obtenido", "Estado"], ["id", "id", "id", "id"])
hr.row_dimensions[K + 1].height = 15
for i, (t, e, o, dato) in enumerate(controles, start=K + 2):
    celda(f"A{i}", t, borde=BORDE)
    celda(f"B{i}", e, "#,##0.00", dato=dato, borde=BORDE)
    celda(f"C{i}", o, "#,##0.00", dato=dato, borde=BORDE)
    celda(f"D{i}", f'=IF(ABS(B{i}-C{i})<1,"OK","REVISAR")', negrita=True, alinear="center", borde=BORDE)
for i in range(K + 1, K + 2 + NK):
    hr.row_dimensions[i].outlineLevel, hr.row_dimensions[i].hidden = 1, True
hr.sheet_properties.outlinePr.summaryBelow = False
FIN_K = K + 2 + NK
for rng_ in (f"D{K + 2}:D{FIN_K - 1}", f"B{K}"):
    hr.conditional_formatting.add(rng_, FormulaRule(formula=[f'COUNTIF($D${K + 2}:$D${FIN_K - 1},"REVISAR")=0'], font=Font(name="Consolas", size=8, bold=True, color="1A8A5A")))
    hr.conditional_formatting.add(rng_, FormulaRule(formula=[f'COUNTIF($D${K + 2}:$D${FIN_K - 1},"REVISAR")>0'], font=Font(name="Consolas", size=8, bold=True, color="C00000")))
celda(f"A{K + 1 + NK + 1}", f"ABC-XYZ calculado con el kardex (12 meses al {CORTE:%d/%m}) · métodos experto/estadístico a revisar: "
      f"{int((plan['Revisar_Metodo'] == 'Sí').sum())} (PLAN: Revisar_Metodo = Sí).").font = GRIS
celda(f"A{K + 1 + NK + 2}", "Cada repuesto y las familias se ven filtrando PLAN. Lo prometido por llave y tipo está en PROMETIDO.").font = GRIS
celda("J9", "Controles", borde=BORDE)
celda("K9", f"=B{K}", negrita=True, alinear="right", borde=BORDE)
hr.conditional_formatting.add("K9", FormulaRule(formula=[f'COUNTIF($D${K + 2}:$D${FIN_K - 1},"REVISAR")=0'], font=Font(name="Consolas", size=8, bold=True, color="1A8A5A")))
hr.conditional_formatting.add("K9", FormulaRule(formula=[f'COUNTIF($D${K + 2}:$D${FIN_K - 1},"REVISAR")>0'], font=Font(name="Consolas", size=8, bold=True, color="C00000")))

# ===== Indicadores (filas 5 a 7) =====
TOT_CAT = FIN_CAT
K_FMT = '"S/" #,##0;[Red]-"S/" #,##0;"-"'
k_txt = lambda ref: f'TEXT({ref}/1000,"#,##0")&"k"'
F_FALTA, E_TVL, E_TATSA, E_TRANS = f"F{r['falta']}", f"E{r['tvl']}", f"E{r['tatsa']}", f"E{r['transinv']}"
tarjetas = [("A", "A", "SALDO AL 31-DIC", f"=E{r['saldo']}", K_FMT, f'="Saldo hoy S/ "&{k_txt(f"B{HOYF}")}&" − falta comprar S/ "&{k_txt(F_FALTA)}', "1F4E78"),
            ("B", "C", "FALTA COMPRAR (" + "–".join(MESES[m] for m in MESES if m >= MES_ACTUAL) + ")", f"={F_FALTA}", K_FMT, f'=TEXT(B{FILA_ACC["COMPRAR"]},"#,##0")&" repuestos · Accion = COMPRAR"', "C00000"),
            ("D", "E", "INVENTARIO AL 31-DIC", f"=E{r['inv']}", K_FMT,
             f'="TVL "&{k_txt(E_TVL)}&" · TATSA "&{k_txt(E_TATSA)}&" · tránsito "&{k_txt(E_TRANS)}', "2E75B6"),
            ("F", "G", f"DESVÍO CONSUMO {TXT_CERRADOS}", f"=IF(ISNUMBER(D{TOT_CAT}),D{TOT_CAT},0)", '+0%;[Red]-0%;"-"',
             f'="Real "&{k_txt(f"C{TOT_CAT}")}&" vs prometido "&{k_txt(f"B{TOT_CAT}")}', "7030A0")]
for c1, c2, etiqueta, valor, fmt, detalle, color in tarjetas:
    lado = Side(style="medium", color=color)
    for fila_, contenido, fuente, alto in [(5, etiqueta, Font(name="Consolas", size=8, bold=True, color="FFFFFF"), 16),
                                          (6, valor, Font(name="Consolas", size=16, bold=True, color=color), 30),
                                          (7, detalle, Font(name="Consolas", size=8, italic=True, color="595959"), 16)]:
        if c1 != c2:
            hr.merge_cells(f"{c1}{fila_}:{c2}{fila_}")
        x = hr[f"{c1}{fila_}"]
        x.value, x.font = contenido, fuente
        x.alignment = Alignment(horizontal="center", vertical="center", wrap_text=False)
        if fila_ == 6:
            x.number_format = fmt
        hr.row_dimensions[fila_].height = alto
        for col_ in range(ord(c1), ord(c2) + 1):
            y = hr[f"{chr(col_)}{fila_}"]
            y.fill = PatternFill("solid", fgColor=color if fila_ == 5 else "F7F9FC")
            y.border = Border(left=lado if chr(col_) == c1 else None, right=lado if chr(col_) == c2 else None,
                              top=lado if fila_ == 5 else None, bottom=lado if fila_ == 7 else None)
hr.conditional_formatting.add("A6", CellIsRule(operator="lessThan", formula=["0"], font=Font(name="Consolas", size=16, bold=True, color="C00000")))

hr.freeze_panes = "A4"
hr.sheet_properties.pageSetUpPr.fitToPage = True
hr.page_setup.orientation, hr.page_setup.fitToWidth, hr.page_setup.fitToHeight = "portrait", 1, 0

wb.calculation.fullCalcOnLoad = True
wb.save(ARCHIVO_EXCEL)
print(f"Excel generado: {ARCHIVO_EXCEL} | hojas {wb.sheetnames} | PLAN {N} llaves")